# Day 16 · Custom Agricultural Object Detection

**Objective:** Download a fruit-and-veggie dataset in YOLO format, train YOLOv8n for 20 epochs, and estimate object counts from predictions.

## 1. Download the Roboflow dataset

The dataset contains annotated fruit and vegetable images with train/validation splits and YOLO-format labels.

In [ ]:
!pip -q install roboflow ultralytics

from getpass import getpass
from roboflow import Roboflow
import yaml
from pathlib import Path

ROBOFLOW_API_KEY = getpass("Enter your Roboflow API key: ")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("projects-gmksb").project("fruit-and-veggie")
dataset = project.version(4).download("yolov8")
DATA_DIR = Path(dataset.location)

with open(DATA_DIR / "data.yaml", "r") as file:
    data_config = yaml.safe_load(file)

class_names = data_config["names"]
if isinstance(class_names, dict):
    class_names = [class_names[i] for i in sorted(class_names)]

print("Dataset:", DATA_DIR)
print("Classes:", class_names)

## 2. Train YOLOv8n for 20 epochs

The `data.yaml` file provides the dataset paths and class names. Training uses the pretrained YOLOv8n weights as a starting point.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=str(DATA_DIR / "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=16,
    device=0 if __import__("torch").cuda.is_available() else "cpu"
)

## 3. Run inference and estimate object counts

Use the trained weights on a validation image. Count the number of detections for each predicted class.

In [ ]:
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

best_model_path = Path(model.trainer.save_dir) / "weights" / "best.pt"
trained_model = YOLO(str(best_model_path))

valid_images = [p for p in (DATA_DIR / "valid").rglob("*")
                if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

if not valid_images:
    raise FileNotFoundError("No validation images found in the dataset.")

image_path = valid_images[0]
predictions = trained_model.predict(
    source=str(image_path),
    conf=0.25,
    device=0 if __import__("torch").cuda.is_available() else "cpu"
)

class_counts = Counter()
for box in predictions[0].boxes:
    class_id = int(box.cls.item())
    class_counts[trained_model.names[class_id]] += 1

print("Estimated object counts:")
for class_name, count in class_counts.items():
    print(f"{class_name}: {count}")

annotated = predictions[0].plot()
plt.figure(figsize=(10, 7))
plt.imshow(annotated[:, :, ::-1])
plt.title("Custom YOLOv8n Predictions")
plt.axis("off")
plt.show()

### Day 16 · Key takeaway

Fine-tuning adapts a pretrained detector to the dataset's classes. Predicted bounding boxes can then be grouped by class to estimate fruit and vegetable counts in an image.